In [1]:
import pandas as pd

In [2]:
DATASET = '../../../datasets/libra/realdata/libra_380K.csv'

df = pd.read_csv(DATASET)
# df = df.iloc[:500000]
df

,id_source,id_destination,cum_amount,nr_transactions,nr_alerts,nr_reports
0,0,349867,1400.00,1,0,0
1,0,351343,4000.00,1,0,0
2,0,350363,30560.00,4,0,0
3,0,349658,5200.00,2,0,0
4,1,351895,646.00,1,0,0
...,...,...,...,...,...,...
597160,385079,193633,458.00,1,0,0
597161,385081,356492,9.88,2,0,0
597162,385086,240249,12882.63,1,0,0
597163,385091,233659,230000.00,5,0,0


In [3]:
df['nr_alerts'].unique()

array([0, 1, 2, 3, 5, 6, 4])

In [4]:
import numpy as np

n_steps = 90
df["step"] = np.random.randint(0, n_steps, size=len(df))

df = df.sort_values("step").reset_index(drop=True)

In [5]:
df

,id_source,id_destination,cum_amount,nr_transactions,nr_alerts,nr_reports,step
0,89467,353896,58000.00,2,0,0,0
1,371758,240513,4000.00,1,0,0,0
2,359085,297896,510.06,11,0,0,0
3,25800,348949,74.56,1,0,0,0
4,351977,338394,2905.00,1,0,0,0
...,...,...,...,...,...,...,...
597160,27548,368267,390.00,1,0,0,89
597161,362453,215043,2142.00,1,0,0,89
597162,27540,371106,900.00,3,0,0,89
597163,351555,298257,24100.00,1,0,0,89


In [6]:
df.columns

Index(['id_source', 'id_destination', 'cum_amount', 'nr_transactions',
       'nr_alerts', 'nr_reports', 'step'],
      dtype='object')

In [7]:
from scipy.sparse import lil_matrix

max_account_id = max(df.id_source.max(),
                     df.id_destination.max()) + 1

print(max_account_id)

matrix = lil_matrix((max_account_id, max_account_id), dtype=int)

385100


In [8]:
from collections import defaultdict

balances = defaultdict(dict)   # {account: {step: balance}}
fraud_account = set()
users_set = set()

fraud = 0

for row in df.itertuples(index=False):
    source = int(row.id_source)
    destination = int(row.id_destination)
    step = int(row.step)
    amount = round(float(row.cum_amount), 3)  

    # Update sparse matrix
    matrix[source, destination] += amount

    # Initialize balances if user appears for the first time
    if source not in balances:
        balances[source] = {step: 0}  # First transaction, balance starts at 0
    if destination not in balances:
        balances[destination] = {step: 0}

    # Get the previous balance (default to 0 if first transaction)
    prev_balance_source = max(balances[source].values(), default=0)
    prev_balance_destination = max(balances[destination].values(), default=0)

    # Update balances for this step
    balances[source][step] = round(prev_balance_source - amount, 3)  # Source loses money
    balances[destination][step] = round(prev_balance_destination + amount, 3)  # Destination gains money

In [9]:
balances[11111]

{0: -1104.0}

In [10]:
# Create DataFrame
balance_df = pd.DataFrame.from_dict(balances, orient='index').T

del balances

# Identify the full range of steps (days)
full_range = range(int(balance_df.index.min()), int(balance_df.index.max()) + 1)

# Reindex to include all steps, then forward fill
balance_df = balance_df.reindex(full_range).fillna(method='ffill').fillna(method='bfill')

/var/folders/s6/4jprwbqj4012tkll2ww6r5hr0000gq/T/ipykernel_8683/1547335081.py:10: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  balance_df = balance_df.reindex(full_range).fillna(method='ffill').fillna(method='bfill')


In [11]:
df = balance_df

del balance_df

In [12]:
columns_as_int = []
for col in df.columns:
    columns_as_int.append(int(str(col).strip()))

columns_as_int.sort()

df = df[columns_as_int]


In [13]:
df

,0,1,2,3,4,5,6,7,8,9,...,385090,385091,385092,385093,385094,385095,385096,385097,385098,385099
0,-4000.0,-646.0,-60.49,-1884.0,-111.0,-456.2,-2975.0,-163000.0,-1726.0,-11523.86,...,250.0,-230000.0,5000.0,2001000.0,104189.81,18000.0,200.0,-389904.0,50.0,50.0
1,-4000.0,-646.0,-60.49,-1884.0,-111.0,-456.2,-2975.0,-163000.0,-1726.0,-11523.86,...,250.0,-230000.0,5000.0,2001000.0,104189.81,18000.0,200.0,-389904.0,50.0,50.0
2,-4000.0,-646.0,-60.49,-1884.0,-111.0,-456.2,-2975.0,-163000.0,-1726.0,-11523.86,...,250.0,-230000.0,5000.0,2001000.0,104189.81,18000.0,200.0,-389904.0,50.0,50.0
3,-4000.0,-646.0,-60.49,-1884.0,-111.0,-456.2,-2975.0,-163000.0,-1726.0,-11523.86,...,250.0,-230000.0,5000.0,2001000.0,104189.81,18000.0,200.0,-389904.0,50.0,50.0
4,-4000.0,-646.0,-60.49,-1884.0,-111.0,-456.2,-2975.0,-163000.0,-1726.0,-11523.86,...,250.0,-230000.0,5000.0,2001000.0,104189.81,18000.0,200.0,-389904.0,50.0,50.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,-5400.0,-646.0,-60.49,-1884.0,-111.0,-456.2,-2975.0,-163000.0,-1726.0,-11523.86,...,250.0,0.0,5000.0,2001000.0,104189.81,18000.0,200.0,-389904.0,50.0,50.0
86,-5400.0,-646.0,-60.49,-1884.0,-111.0,-456.2,-2975.0,-163000.0,-1726.0,-11523.86,...,250.0,0.0,5000.0,2001000.0,104189.81,18000.0,200.0,-389904.0,50.0,50.0
87,-5400.0,-646.0,-60.49,-1884.0,-111.0,-456.2,-2975.0,-163000.0,-1726.0,-11523.86,...,250.0,0.0,5000.0,2001000.0,104189.81,18000.0,200.0,-389904.0,50.0,50.0
88,-5400.0,-646.0,-60.49,-1884.0,-111.0,-456.2,-2975.0,-163000.0,-1726.0,-11523.86,...,250.0,0.0,5000.0,2001000.0,104189.81,18000.0,200.0,-389904.0,50.0,50.0


In [14]:
# import matplotlib.pyplot as plt

# # Plot the balance evolution for all users (columns) as separate lines
# # plt.figure(figsize=(15,10), dpi= 300)
# plt.figure(figsize=(15,10))

# # Plot each column (account balance) as a line
# for user in balance_df.columns:
#     plt.plot(balance_df.index, balance_df[user], label=f"User {user}", alpha=0.6)

# # Add title and labels
# plt.xlabel("Days", fontsize=22)
# plt.ylabel("Balance", fontsize=22)

# # Rotate x-axis labels for better visibility
# plt.tick_params(axis='x', labelsize=22)
# plt.tick_params(axis='y', labelsize=22)

# # Display the plot
# plt.tight_layout()
# plt.show()

In [15]:
Y = df.T.to_numpy()

In [16]:
l = [35.5, 35.4, 35.3, 35]

In [17]:
Theta_dict = {}

In [18]:
ROWS = Y.shape[0]
ROWS

385100

In [19]:
from SQUIC_functions import *
from scipy import sparse

: 

In [ ]:
data_nnz = []
data_nnzr = []
data_time = []
data_sym = []

for rho in l:
    Theta, _, computation_time = compute_squic(Y, rho)
    computation_time = round(computation_time, 2)
    print(f"time required: {computation_time} seconds")

    Theta = sparse.csr_matrix(Theta) # store Theta as sparse
    Theta_dict[rho] = Theta

    nonz, nonz_r = nnz_sparse(Theta_dict[rho], ROWS)

    print("Checking if symmetric")
    if check_symmetric_sparse(Theta_dict[rho]):
        print(f"✅ Matrix is symmetric per rho {rho}")
        data_sym.append('Yes')
    else:
        print(f"❌ Matrix is not symmetric per rho {rho}")
        data_sym.append('No')

    data_nnz.append(nonz)
    data_nnzr.append(nonz_r)
    data_time.append(computation_time)

    if nonz == ROWS:
        print(f"For rho = {rho} just element on the diagonal")
        sparsity_pattern(Theta_dict[rho])
    elif nonz == ROWS * ROWS:
        print(f"For rho = {rho} big dense matrix")
        print(f"nnz = {nonz}")
        print(f"nnz/row = {nonz_r}")
        sparsity_pattern(Theta_dict[rho])
    else:
        print(f"For rho = {rho} there are some nnz")
        print(f"nnz = {nonz}")
        print(f"nnz/row = {nonz_r}")
        sparsity_pattern(Theta_dict[rho])


table_squic_norm = [
    ["NNZ"] + data_nnz,
    ["NNZ/Row"] + data_nnzr,
    ["Time (s)"] + data_time,
    ["Symmetric"] + data_sym
]

----------------------------------------------------------------
                     SQUIC Version 1.0                         
----------------------------------------------------------------
Input Matrices
 nnz(X0)/p:   1.000000e+00
 nnz(W0)/p:   1.000000e+00
 nnz(M)/p:    ignored
 Y:           385100 x 90 
Runtime Configs   
 Sample Cov.:   Deterministic
 CordDec Vers:  Collective coordinate descent update 
 Inversion:     Approx. block Neumann series
 Fact. Routine: CHOLMOD
Parameters       
 verbose:     1 
 lambda:      3.550000e+01 
 max_iter:    100 
 term_tol:    1.000000e-03 
 inv_tol:     1.000000e-04 
 threads:     12 

#SQUIC Started 


# Squic-Fit

In [28]:
W_dict = {}

data_nnz = []
data_nnzr = []
data_time = []
data_sym = []

for rho in l:
    W, end_time = squic_fit_matrix_sparse(Y, l=rho, matrix=matrix)
    W_dict[rho] = sparse.csr_matrix(W)
    end_time = round(end_time, 2)
    print(f"required time: {end_time}")

    nnz, nnz_r = nnz_sparse(W_dict[rho], ROWS)
    print(f"nnz = {nnz} per rows = {nnz_r}")

    sparsity_pattern(W_dict[rho])

    if check_symmetric_sparse(W_dict[rho]):
        print(f"✅ Matrix is symmetric per rho {rho}")
        data_sym.append("Yes")
    else:
        print(f"❌ Matrix is not symmetric per rho {rho}")
        data_sym.append("No")

    data_nnz.append(nnz)
    data_nnzr.append(nnz_r)
    data_time.append(end_time)

table_fit_norm = [
    ["NNZ"] + data_nnz,
    ["NNZ/Row"] + data_nnzr,
    ["Time (s)"] + data_time,
    ["Symmetric"] + data_sym
]


----------------------------------------------------------------
                     SQUIC Version 1.0                         
----------------------------------------------------------------
Input Matrices
 nnz(X0)/p:   1.000000e+00
 nnz(W0)/p:   1.000000e+00
 nnz(M)/p:    3.018489e+00
 Y:         385100 x 90 
Runtime Configs   
 Sample Cov.:   Deterministic
 CordDec Vers:  Collective coordinate descent update 
 Inversion:     Approx. block Neumann series
 Fact. Routine: CHOLMOD
Parameters       
 verbose:     1 
 lambda:      3.550000e+01 
 max_iter:    100 
 term_tol:    1.000000e-03 
 inv_tol:     1.000000e-04 
 threads:     12 

#SQUIC Started 


: 